# 02 · Dataset exploration

Datasets are **verified** first (identifiers, licences, splits, sizes) — see `data/README.md`
and re-run the checks with `python scripts/probe_datasets.py`.

| Dataset | Where | Used for |
|---|---|---|
| QMSum (official, MIT) | GitHub `Yale-LILY/QMSum`, jsonl | main summarisation benchmark |
| AMI (CC-BY-4.0) | HF `edinburghcstr/ami`, **streaming text only** | meeting structure |
| DialogSum (CC-BY-NC-SA-4.0) | HF `knkarthick/dialogsum` | optional warm-up |

Large corpora are **never** downloaded in full (AMI full ≈ 15 GB → we stream 2 text meetings).

In [ ]:
print('[1/4] Downloading QMSum (~17 MB, resumable)')
!python -u scripts/download_data.py

In [ ]:
print('[2/4] Load a validation meeting (unified schema)')
import sys; sys.path.insert(0, '.')
from src.config import load_config, resolve_path
from src.data_loader import load_qmsum

cfg = load_config()
meetings = load_qmsum(resolve_path(cfg, 'data_raw') / 'qmsum_val.jsonl', 'val', limit=3)
m = meetings[0]
print('meeting_id    :', m['meeting_id'])
print('title         :', m['title'][:100])
print('date          :', m['date'], '(QMSum carries no dates -> Not specified)')
print('participants  :', len(m['participants']), 'speakers')
print('turns         :', len(m['turns']))
print('reference     :', m['reference_summary'][:200], '...')
print()
print('first 3 turns:')
for t in m['turns'][:3]:
    print(f"  {t['speaker'][:40]:40s} | {t['text'][:80]}")

In [ ]:
print('[3/4] Length distribution of the validation split (token estimates)')
import matplotlib.pyplot as plt
from src.chunking import count_tokens

val = load_qmsum(resolve_path(cfg, 'data_raw') / 'qmsum_val.jsonl', 'val', limit=50)
lengths = [count_tokens(' '.join(t['text'] for t in m['turns'])) for m in val]
plt.figure(figsize=(7, 3.5))
plt.hist(lengths, bins=20, color='#4C72B0')
plt.xlabel('estimated tokens per meeting'); plt.ylabel('meetings')
plt.title('QMSum val: meeting lengths (n=%d)' % len(val))
plt.tight_layout(); plt.show()
print('mean %.0f tokens -> a meeting needs ~%d chunks of 700 tokens'
      % (sum(lengths)/len(lengths), round(sum(lengths)/len(lengths)/700)))

In [ ]:
print('[4/4] Optional: stream 1 AMI meeting (text only, no audio download)')
from src.data_loader import load_ami_streaming
try:
    ami = load_ami_streaming(n_meetings=1)
    m = ami[0]
    print(m['meeting_id'], '| turns:', len(m['turns']), '| speakers:', m['participants'])
    print('turn0:', m['turns'][0])
except Exception as e:
    print('AMI streaming skipped/failed (network?):', type(e).__name__, e)